# 22 - k vecinos mas cercanos (k-NN): flujo completo de clasificacion

## Objetivos de aprendizaje

En esta sesion aprenderas a:

1. Recordar de forma breve que es ciencia de datos.
2. Entender la idea de distancia y su generalizacion.
3. Ver por que normalizar datos es clave en k-NN.
4. Aplicar k-NN desde cero en un flujo train/test.
5. Evaluar accuracy para varios valores de `k`.
6. Interpretar la matriz de confusion.
7. Pensar decisiones de modelado, no solo ejecutar codigo.


## Ruta de la sesion

1. Recordatorio de ciencia de datos.
2. Intuicion de k-NN.
3. Distancia (euclidiana y Minkowski).
4. Por que normalizar.
5. Pasos del algoritmo y variaciones.
6. Flujo completo con dataset `Wine`.
7. Accuracy para distintos `k`.
8. Matriz de confusion e interpretacion.
9. Ejercicios de pensamiento.


## 1) Recordatorio rapido: ciencia de datos

Ciencia de datos no es solo entrenar modelos.
Es un proceso para resolver decisiones con datos:

1. Definir problema y objetivo.
2. Entender y preparar datos.
3. Modelar.
4. Evaluar si realmente ayuda a decidir.
5. Implementar y monitorear.

Hoy nos enfocamos en la fase de modelado + evaluacion con un clasificador sencillo: k-NN.


## 2) Intuicion de k-NN

k-NN (k nearest neighbors) predice la clase de un punto nuevo mirando los `k` ejemplos mas cercanos en el entrenamiento.

- Si la mayoria de vecinos cercanos es clase A, predice A.
- Si la mayoria es clase B, predice B.

Es un modelo basado en similitud.
Por eso la nocion de distancia es central.


## 3) Distancia y su generalizacion

Distancia euclidiana entre dos puntos `x` y `z`:

$d(x, z) = \sqrt{\sum_{j=1}^{m} (x_j - z_j)^2}$

Generalizacion de Minkowski:

$d_p(x, z) = \left(\sum_{j=1}^{m} |x_j - z_j|^p\right)^{1/p}$

- `p = 1`: Manhattan.
- `p = 2`: Euclidiana.

Elegir metrica cambia quienes son los "vecinos mas cercanos".


## 4) Por que normalizar es importante

Si una variable tiene escala gigante y otra escala chica, la grande domina la distancia.

Ejemplo:

- variable A en rango [0, 1]
- variable B en rango [0, 1000]

Sin normalizar, diferencias en B pesan mucho mas que en A.
En k-NN esto puede distorsionar la vecindad y empeorar predicciones.

Por eso es comun usar `StandardScaler` (media 0, desviacion estandar 1).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


## 5) Datos: dataset Wine

Usaremos `Wine`, un dataset clasico multiclase (3 tipos de vino) con mediciones quimicas.
Es bueno para mostrar el impacto de normalizar porque las variables tienen escalas muy distintas.


In [ ]:
wine = load_wine()
X = wine.data
y = wine.target
feature_names = wine.feature_names
target_names = wine.target_names

print("Forma de X:", X.shape)
print("Forma de y:", y.shape)
print("Clases:", target_names)
print("Primeras 5 variables:", feature_names[:5])


In [ ]:
# Revisamos rapidamente diferencias de escala por variable.
mins = X.min(axis=0)
maxs = X.max(axis=0)
ranges = maxs - mins

print("Rango minimo entre variables:", ranges.min())
print("Rango maximo entre variables:", ranges.max())
print("\nEjemplos de rangos por variable:")
for i in [0, 3, 6, 12]:
    print(f"{feature_names[i]:>28}: min={mins[i]:8.3f}, max={maxs[i]:8.3f}, rango={ranges[i]:8.3f}")


## 6) Pasos del algoritmo k-NN

Para clasificacion:

1. Elegir `k` y una metrica de distancia.
2. Para cada punto nuevo, calcular distancia a todos los puntos de entrenamiento.
3. Tomar los `k` vecinos mas cercanos.
4. Votar la clase mayoritaria (o ponderar por distancia).
5. Asignar clase predicha.

Costo: puede ser alto en prediccion si hay muchos datos, porque compara contra todo el entrenamiento.


## 7) Variaciones comunes

- `weights='uniform'`: todos los vecinos pesan igual.
- `weights='distance'`: vecinos mas cercanos pesan mas.
- `metric='minkowski', p=1`: distancia Manhattan.
- `metric='minkowski', p=2`: distancia Euclidiana.
- `k` pequeno: baja sesgo, alta varianza (mas sensible a ruido).
- `k` grande: mayor sesgo, menor varianza (frontera mas suave).


## 8) Flujo completo: train/test

Dividimos datos para simular evaluacion honesta en datos no vistos.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print("Train:", X_train.shape, "Test:", X_test.shape)


In [ ]:
def accuracy_para_varios_k(
    X_train,
    X_test,
    y_train,
    y_test,
    k_values,
    usar_escalado,
    metric="minkowski",
    p=2,
    weights="uniform",
):
    accuracies = []
    for k in k_values:
        if usar_escalado:
            modelo = Pipeline([
                ("scaler", StandardScaler()),
                (
                    "knn",
                    KNeighborsClassifier(n_neighbors=k, metric=metric, p=p, weights=weights),
                ),
            ])
        else:
            modelo = KNeighborsClassifier(n_neighbors=k, metric=metric, p=p, weights=weights)

        modelo.fit(X_train, y_train)
        pred = modelo.predict(X_test)
        accuracies.append(accuracy_score(y_test, pred))

    return np.array(accuracies)


In [ ]:
k_values = list(range(1, 32, 2))  # usamos impares para reducir empates

acc_sin_escalar = accuracy_para_varios_k(
    X_train,
    X_test,
    y_train,
    y_test,
    k_values,
    usar_escalado=False,
)

acc_con_escalar = accuracy_para_varios_k(
    X_train,
    X_test,
    y_train,
    y_test,
    k_values,
    usar_escalado=True,
)


In [ ]:
plt.figure(figsize=(8.5, 5))
plt.plot(k_values, acc_sin_escalar, marker="o", label="Sin escalar")
plt.plot(k_values, acc_con_escalar, marker="s", label="Con StandardScaler")
plt.xlabel("k")
plt.ylabel("accuracy en test")
plt.title("k-NN en Wine: efecto de k y de normalizacion")
plt.xticks(k_values)
plt.ylim(0.5, 1.02)
plt.grid(alpha=0.25)
plt.legend()
plt.show()


In [ ]:
idx_best_sin = int(np.argmax(acc_sin_escalar))
idx_best_con = int(np.argmax(acc_con_escalar))

best_k_sin = k_values[idx_best_sin]
best_k_con = k_values[idx_best_con]

print(f"Mejor sin escalar: k={best_k_sin}, accuracy={acc_sin_escalar[idx_best_sin]:.3f}")
print(f"Mejor con escalar: k={best_k_con}, accuracy={acc_con_escalar[idx_best_con]:.3f}")


## 9) Que mide accuracy

Accuracy es:

$$\text{accuracy} = \frac{\text{predicciones correctas}}{\text{total de predicciones}}$$

Interpretacion:

- Si accuracy = 0.90, 90% de ejemplos del test se clasificaron bien.
- Es facil de entender, pero puede ocultar errores importantes por clase.

Por eso la complementamos con matriz de confusion.


In [ ]:
# Entrenamos ambos mejores modelos para comparar sus matrices.
model_best_sin = KNeighborsClassifier(n_neighbors=best_k_sin)
model_best_sin.fit(X_train, y_train)
pred_best_sin = model_best_sin.predict(X_test)

model_best_con = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=best_k_con)),
])
model_best_con.fit(X_train, y_train)
pred_best_con = model_best_con.predict(X_test)

cm_sin = confusion_matrix(y_test, pred_best_sin)
cm_con = confusion_matrix(y_test, pred_best_con)


In [ ]:
def plot_confusion(ax, cm, title, labels):
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="black")

    return im


fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
im1 = plot_confusion(axes[0], cm_sin, f"Sin escalar (k={best_k_sin})", target_names)
im2 = plot_confusion(axes[1], cm_con, f"Con escalar (k={best_k_con})", target_names)

fig.colorbar(im2, ax=axes.ravel().tolist(), shrink=0.9)
plt.tight_layout()
plt.show()


## 10) Como leer la matriz de confusion

En esta notebook usamos:

- filas = clase real,
- columnas = clase predicha.

Lectura:

1. La diagonal principal son aciertos.
2. Fuera de la diagonal estan las confusiones.
3. Si una fila tiene muchos errores fuera de diagonal, esa clase se esta confundiendo seguido.

Accuracy te da resumen global.
Matriz de confusion te dice en que clases falla el modelo.


## 11) Variaciones rapidas: metrica y peso de vecinos

Probamos tres configuraciones sobre el mismo split y con escalado.


In [ ]:
experimentos = [
    ("euclidiana_uniforme", "minkowski", 2, "uniform"),
    ("manhattan_uniforme", "minkowski", 1, "uniform"),
    ("euclidiana_distance", "minkowski", 2, "distance"),
]

for nombre, metric, p, weights in experimentos:
    accs = accuracy_para_varios_k(
        X_train,
        X_test,
        y_train,
        y_test,
        k_values,
        usar_escalado=True,
        metric=metric,
        p=p,
        weights=weights,
    )
    idx = int(np.argmax(accs))
    print(f"{nombre:>22}: mejor k={k_values[idx]:2d}, accuracy={accs[idx]:.3f}")


No hay una configuracion universalmente mejor.
La eleccion depende del problema, escala de datos, ruido y objetivo de negocio.


## 12) Ejercicios de pensamiento (no copiar y pegar)

Escribe primero tu razonamiento en texto. Luego valida con codigo.


### Ejercicio 1: elegir metrica con argumento

Supongamos que tus variables tienen outliers fuertes en algunas dimensiones.

1. Que distancia elegirias entre Manhattan y Euclidiana.
2. Justifica en terminos de sensibilidad a diferencias grandes.
3. Diseña un mini experimento para comprobar tu hipotesis.


In [ ]:
# Escribe aqui tu experimento para Ejercicio 1.


### Ejercicio 2: bias-varianza con k extremo

Sin ejecutar, predice que esperas en train y test para:

- `k = 1`
- `k = 31`

Luego mide ambos accuracies y explica si coinciden con tu hipotesis.


In [ ]:
# Mide accuracy en train y test para k=1 y k=31.


### Ejercicio 3: fuga de informacion (data leakage)

Pregunta conceptual:

1. Que pasa si escalas TODO el dataset antes de dividir train/test.
2. Por que eso puede inflar metricas.
3. Como lo evitas en un flujo reproducible.


In [ ]:
# Escribe aqui una comparacion entre flujo correcto e incorrecto.


### Ejercicio 4: mas alla de accuracy

Imagina que confundir clase A con clase B cuesta mucho dinero, pero confundir B con C cuesta poco.

1. Por que accuracy sola no es suficiente.
2. Como usarias la matriz de confusion para decidir.
3. Que metrica adicional propones (precision/recall/F1 por clase) y por que.


In [ ]:
# Calcula aqui precision, recall y F1 por clase para tu mejor modelo.


### Ejercicio 5: diseña una recomendacion tecnica

Con base en tus resultados, redacta una recomendacion breve para un equipo no tecnico:

1. que valor de `k` elegir,
2. si escalar o no,
3. principal riesgo operativo del modelo,
4. siguiente experimento que harias antes de desplegar.


In [ ]:
# Escribe aqui tu recomendacion (puede ser en comentarios o texto).


## 13) Cierre

Ideas clave:

1. k-NN clasifica por cercania entre ejemplos.
2. La distancia elegida define la nocion de vecino.
3. Normalizar suele ser critico para que ninguna variable domine injustamente.
4. `k` controla el equilibrio entre sensibilidad y suavizado.
5. Accuracy resume rendimiento global, pero la matriz de confusion revela donde falla.
